# PA2: SFT + DPO on Qwen2.5-0.5B - training, generation, evaluation prep

Everything except the model-based judging runs here, in one notebook: data
preparation, supervised fine-tuning, two from-scratch DPO runs, the TRL/LoRA
persona run, generation on the fixed evaluation sets, and the blinded pairwise
files.

### Why judging is a second notebook

The manual describes a three-notebook workflow, but as a recommendation - "a
practical workflow is" - and section 13.1 itself refers to "the
training/generation notebook" in the singular. So merging training and
generation is allowed.

Judging cannot be merged, for a hard technical reason rather than a stylistic
one: `kaggle_benchmarks` is **not importable in a GPU notebook**. Verified on
this exact image:

```text
=== is kaggle_benchmarks usable in a GPU notebook? ===
kaggle_benchmarks NOT importable here: No module named 'kaggle_benchmarks'
```

The judge models live in Kaggle's benchmark/model-proxy environment, which is a
different notebook type. So `02_judge_kaggle.ipynb` is created from
<https://www.kaggle.com/benchmarks/tasks/new> and reads the pairwise files this
notebook produces. That also satisfies the manual's rule that judging runs from
saved pairwise files only and never regenerates model outputs.

### Required Kaggle settings

- **Accelerator**: `GPU T4 x2`
- **Internet**: `On`
- **Input**: the dataset holding this project (and, to continue an earlier run, that run's output)

### This notebook is restartable

A full run is 6-8 hours and Kaggle sessions do die. Three mechanisms mean a
death costs minutes, not hours:

| Mechanism | What it does |
|---|---|
| **Resume checkpoints** | SFT and DPO write model + optimizer + scheduler + RNG + data position every `resume_every` steps, atomically. A rerun continues from there. |
| **Time budget** | Each stage stops *voluntarily* before the session limit, saving a checkpoint first, rather than being killed mid-step. |
| **Stage ledger** | `outputs/pipeline_state.json` records what finished. A rerun skips completed stages, continues paused ones, retries failed ones. |

Nothing here raises out of a cell: a failed stage is recorded and the notebook
runs on to the end, so the commit always completes and the outputs always save.

**To continue a previous run:** set `PREV_OUTPUT` below to that run's output,
and run all. Finished work is restored, not recomputed.

## 1. Settings

Everything you might change is in this one cell.

In [ ]:
# ---- where the code and any previous outputs come from --------------------
# Attach the project dataset under "Add Input". If the path differs, fix it here.
PROJECT_DATASET = "/kaggle/input/pa2-small-model-alignment"
PROJECT_DIR = "/kaggle/working/small-model-alignment"

# To CONTINUE an earlier run: attach that run's output as an input and point
# this at it, e.g. "/kaggle/input/pa2-pipeline-run1". None = start fresh.
PREV_OUTPUT = None

# ---- session time budget --------------------------------------------------
# Kaggle kills a session at its limit with no chance to save. Stages stop
# themselves before that, leaving a resume checkpoint behind. Keep this
# comfortably under the real limit.
SESSION_BUDGET_HOURS = 10.5

# ---- which stages to run --------------------------------------------------
# Completed stages are skipped automatically via the ledger, so you normally
# leave all of these on.
RUN_DATA_PREP   = True
RUN_SFT         = True
RUN_DPO_HH      = True
RUN_DPO_UF      = True
RUN_LORA_DPO    = True
RUN_GENERATION  = True

FORCE_RERUN = []   # e.g. ["sft"] to redo a stage that is already marked complete

# ---- the reference recipe -------------------------------------------------
SEED = 42
BASE_MODEL = "Qwen/Qwen2.5-0.5B"
LORA_BASE_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
SFT_TRAIN, SFT_VAL = 30_000, 1_000
PREF_TRAIN, PREF_VAL = 15_000, 1_000
ALPACA_PROMPTS = 300
SFT_EVAL_EVERY, DPO_EVAL_EVERY = 100, 50
SFT_RESUME_EVERY, DPO_RESUME_EVERY = 150, 100
DPO_EVAL_SUBSET = 300
LORA_EPOCHS = 3

# ---- decoding, shared by every model in a comparison ----------------------
MAX_NEW_TOKENS = 256
REPETITION_PENALTY = 1.1
NO_REPEAT_NGRAM = 0
GEN_BATCH_SIZE = 8

# ---- smoke test -----------------------------------------------------------
# Runs the entire pipeline end to end at ~1% scale in roughly 15-25 minutes.
# Worth doing once before committing to a full overnight run: it exercises every
# stage, every checkpoint path and every file format on the real Kaggle image.
# The resulting models are not meaningful - this checks plumbing, not quality.
SMOKE_TEST = False

if SMOKE_TEST:
    SFT_TRAIN, SFT_VAL = 400, 40
    PREF_TRAIN, PREF_VAL = 200, 40
    ALPACA_PROMPTS = 12
    SFT_EVAL_EVERY, DPO_EVAL_EVERY = 4, 3
    SFT_RESUME_EVERY, DPO_RESUME_EVERY = 4, 3
    DPO_EVAL_SUBSET = 16
    LORA_EPOCHS = 1
    MAX_NEW_TOKENS = 48
    print("SMOKE TEST MODE: tiny sizes, results are not meaningful")

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"


def find_project(dataset_dir, work_dir):
    """Copy the project out of the read-only input dataset into /kaggle/working.

    It has to be writable: every stage writes under ``outputs/`` next to the code.

    The search is recursive because Kaggle does not guarantee where a dataset
    mounts - it has appeared at ``/kaggle/input/<slug>/`` and at
    ``/kaggle/input/datasets/<owner>/<slug>/``. Rather than guess the layout,
    look for the package itself.
    """
    target = Path(work_dir)
    if (target / "smalign").is_dir():
        print(f"using project already at {target}")
        return target

    roots = []
    if dataset_dir and Path(dataset_dir).exists():
        roots.append(Path(dataset_dir))
    if Path("/kaggle/input").is_dir():
        roots.append(Path("/kaggle/input"))
    if Path("/kaggle/working").is_dir():
        roots.append(Path("/kaggle/working"))

    for root in roots:
        for marker in sorted(root.rglob("smalign/__init__.py")):
            candidate = marker.parent.parent
            if (candidate / "submission_adapter.py").is_file():
                print(f"found project at {candidate}")
                shutil.copytree(candidate, target, dirs_exist_ok=True)
                print(f"copied -> {target}")
                return target

    # Nothing found: show what is actually mounted, which is the one thing
    # that makes this fixable without another round trip.
    listing = []
    for root in roots:
        listing.append(f"  {root}/")
        for path in sorted(root.rglob("*"))[:60]:
            depth = len(path.relative_to(root).parts)
            if depth <= 3:
                listing.append(f"    {path.relative_to(root)}{'/' if path.is_dir() else ''}")
    raise FileNotFoundError(
        "Could not find the project (looked for smalign/__init__.py).\n"
        "Attach the dataset holding it: Add Input -> Datasets -> your dataset.\n"
        "What is mounted:\n" + ("\n".join(listing) if listing else "  (nothing)")
    )


def restore_previous(prev, project, subdirs=("outputs", "data/processed", "eval_sets")):
    """Bring a previous run's artifacts in so finished stages are skipped.

    This is what makes a run resumable across sessions: the ledger, the
    checkpoints and the resume files come back, and the pipeline continues where
    it stopped instead of starting over.

    Like ``find_project``, the lookup is recursive. A kernel's output mounts at a
    path Kaggle chooses, and the project sits one level inside it, so the
    directories are located by searching rather than by assuming a layout.
    """
    if not prev:
        print("PREV_OUTPUT not set - starting fresh")
        return

    prev = Path(prev)
    if not prev.exists():
        # The configured path is wrong or Kaggle mounted it elsewhere. Rather
        # than silently start a six-hour run from zero, look for a previous
        # run's ledger anywhere under /kaggle/input and use that.
        print(f"PREV_OUTPUT {prev} does not exist - searching /kaggle/input for a previous run")
        found = None
        if Path("/kaggle/input").is_dir():
            for marker in sorted(Path("/kaggle/input").rglob("outputs/pipeline_state.json")):
                found = marker.parent.parent
                break
        if found is None:
            print("  no previous run found; starting fresh")
            print("  available inputs:",
                  [str(p) for p in Path("/kaggle/input").glob("*")]
                  if Path("/kaggle/input").is_dir() else "none")
            return
        prev = found
        print(f"  using {prev}")

    restored = []
    for sub in subdirs:
        source = None
        direct = prev / sub
        if direct.is_dir():
            source = direct
        else:
            # Find it anywhere underneath, preferring the shallowest match.
            leaf = Path(sub).name
            matches = [m for m in prev.rglob(leaf) if m.is_dir() and m.match(f"*/{sub}")]
            if not matches:
                matches = [m for m in prev.rglob(leaf) if m.is_dir()]
            if matches:
                source = min(matches, key=lambda m: len(m.parts))
        if source:
            destination = Path(project) / sub
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=True)
            restored.append(f"{sub} <- {source}")

    for line in restored:
        print("restored", line)
    if not restored:
        print(f"nothing to restore from {prev}")
    else:
        ledger_file = Path(project) / "outputs" / "pipeline_state.json"
        if ledger_file.exists():
            state = json.loads(ledger_file.read_text())
            done = [k for k, v in state.get("stages", {}).items() if v.get("status") == "completed"]
            print(f"ledger restored: {len(done)} stage(s) already complete -> {', '.join(done) or 'none'}")


def disk_report(root="/kaggle/working"):
    try:
        total, used, free = shutil.disk_usage(root)
    except OSError:
        return
    print(f"disk {root}: {used/2**30:.1f} GiB used, {free/2**30:.1f} GiB free of {total/2**30:.1f}")
    rows = []
    for path in sorted(Path(root).rglob("*")):
        if path.is_dir() and path.name in {"best", "final", "adapter"}:
            rows.append((sum(f.stat().st_size for f in path.rglob("*") if f.is_file()), path))
        elif path.is_file() and path.name == "resume.pt":
            rows.append((path.stat().st_size, path))
    for size, path in sorted(rows, reverse=True)[:12]:
        print(f"  {size/2**30:6.2f} GiB  {path}")

## 2. Dependencies

Kaggle's image already carries a CUDA-matched PyTorch, Transformers, Datasets
and PEFT, so none of those are touched - reinstalling PyTorch wastes time and
can pull a build that cannot see the GPU.

Two deliberate choices:

- **`trl` is the only real install.** It is the one package the image lacks.
- **Transformers is *not* pinned below 5.** The manual suggests
  `"transformers>=4.43.0,<5"`, but this image ships 5.16.1 against torch 2.11,
  and forcing a downgrade risks breaking a working CUDA stack. Every call this
  project makes was verified against 5.16.1 on this image, and the model loader
  handles the v4/v5 `torch_dtype` to `dtype` rename either way. This is a
  deviation from the guide and is recorded as such in the report.

The `torchao` uninstall avoids a known conflict with PEFT/LoRA loading.

In [ ]:
!python -m pip uninstall -y -q torchao 2>/dev/null
!python -m pip install -q "trl>=0.11.0" pytest 2>&1 | tail -3

import importlib
for pkg in ["torch", "transformers", "datasets", "peft", "trl", "accelerate"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  {pkg:14s} MISSING ({type(e).__name__})")

## 3. Project, previous outputs, and the stage ledger

In [ ]:
project = find_project(PROJECT_DATASET, PROJECT_DIR)
%cd {project}
sys.path.insert(0, str(project))

restore_previous(PREV_OUTPUT, project)

from smalign.pipeline import Ledger, print_ledger, run_stage
from smalign.utils import read_json

ledger = Ledger("outputs/pipeline_state.json")

SESSION_START = time.time()
DEADLINE = SESSION_START + SESSION_BUDGET_HOURS * 3600

def budget_left() -> int:
    """Seconds a stage may use before it should checkpoint and pause."""
    return max(int(DEADLINE - time.time()), 60)

# Cheap, derived stages are regenerated every run: after a paused stage is
# resumed there are new metrics, so a cached plot would be stale.
ALWAYS_RERUN = {"plots"}

def stage(name, command, produces=(), depends_on=(), quiet_minutes=45):
    """Run a stage with a hard cap and a silence cap.

    Every stage here prints progress periodically, so a long gap means it is
    wedged rather than working. Killing it turns a hung session into a recorded
    failure that the next run retries from its checkpoint.
    """
    return run_stage(name, command, list(produces), list(depends_on), ledger=ledger,
                     force=(name in FORCE_RERUN or name in ALWAYS_RERUN),
                     timeout=budget_left() + 900,
                     quiet_timeout=quiet_minutes * 60)

print()
print_ledger(ledger)
print(f"session budget {SESSION_BUDGET_HOURS}h -> {budget_left()/3600:.2f}h available")
disk_report()

## 4. Environment check

Task 1: confirm two T4s are visible and record the versions the run actually used.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch
from smalign.utils import describe_environment
info = describe_environment()
print(f"\ntorch {info['torch']} | cuda {info['cuda_available']} | GPUs {info['device_count']}")
for d in info["devices"]:
    print(f"  cuda:{d['index']}  {d['name']}  {d['total_memory_gib']} GiB")
if info["device_count"] < 2:
    print("\nWARNING: fewer than two GPUs. Training still works (the frozen DPO "
          "reference shares one GPU) but set 'GPU T4 x2' to follow the recipe.")

## 5. Public adapter tests

The graded interface, plus the edge cases the manual warns the public suite does
not cover: truncation boundaries, tokenizers with no pad or EOS id, prompts
longer than `max_length`, row ordering, and real optimization steps.

`tests/test_resume.py` is the important one for an unattended run. It trains a
tiny model straight through, then trains it again with an interruption, and
asserts the **weights match**. Resuming has to be equivalent to never having
stopped - not merely "it restarts without crashing".

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/ -x 2>&1 | tail -15

## 6. Fixed data splits

| Split | Size | Source |
|---|---|---|
| SFT train / val | 30,000 / 1,000 | UltraChat-style single-turn pairs |
| HH-RLHF train / val | 15,000 / 1,000 | Helpful/harmless pairs, single-turn only |
| UltraFeedback train / val | 15,000 / 1,000 | General preference pairs |
| AlpacaEval | 300 prompts, seed 42 | Fixed subset, saved to `eval_sets/` |
| Persona train / val | ~200 / 20 | Provided persona preference data |

The large corpora stream and keep the first N rows that pass the filters, which
is deterministic and avoids downloading gigabytes to select 30k rows. Multi-turn
HH conversations are dropped, since the prompt format here is single-turn.

In [ ]:
ALPACA_EVAL_PATH = f"eval_sets/alpaca_eval_{ALPACA_PROMPTS}_seed{SEED}.json"
HH_EVAL_PATH = "eval_sets/hh_rlhf_eval_300_seed42.json"
PERSONA_EVAL_PATH = "eval_sets/persona_style_eval.json"

if RUN_DATA_PREP:
    stage(
        "data_prep",
        ["python", "scripts/prepare_data.py", "--stages", "all",
         "--sft_train", str(SFT_TRAIN), "--sft_val", str(SFT_VAL),
         "--pref_train", str(PREF_TRAIN), "--pref_val", str(PREF_VAL),
         "--alpaca_prompts", str(ALPACA_PROMPTS), "--seed", str(SEED)],
        produces=["data/processed/sft_train.json", "data/processed/hh_train.json",
                  "data/processed/uf_train.json", "data/processed/persona_train.json",
                  ALPACA_EVAL_PATH],
    )

In [ ]:
# Look at the data before training on it.
for name, path in [("SFT train", "data/processed/sft_train.json"),
                   ("HH train", "data/processed/hh_train.json"),
                   ("UltraFeedback train", "data/processed/uf_train.json"),
                   ("Persona train", "data/processed/persona_train.json"),
                   ("AlpacaEval subset", ALPACA_EVAL_PATH),
                   ("HH eval set", HH_EVAL_PATH)]:
    if Path(path).exists():
        rows = read_json(path)
        print(f"{name:22s} {len(rows):>6,} rows   keys: {sorted(rows[0])}")
    else:
        print(f"{name:22s} MISSING ({path})")

if Path("data/processed/hh_train.json").exists():
    row = read_json("data/processed/hh_train.json")[0]
    print("\n" + "=" * 78 + "\nOne HH preference pair\n" + "=" * 78)
    print("instruction:", row["instruction"][:250])
    print("chosen:     ", row["chosen"][:220])
    print("rejected:   ", row["rejected"][:220])

### The prompt format

One template for SFT, both DPO runs, and generation. If they disagreed, the
response masks and the implicit DPO rewards would stop lining up. The check
below confirms the mask really covers the response and nothing else.

In [ ]:
from transformers import AutoTokenizer
from smalign.batching import build_lm_batch
from smalign.prompts import format_prompt

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("prompt template:", repr(format_prompt("{instruction}")))
print(f"eos {tokenizer.eos_token!r} id {tokenizer.eos_token_id}")

batch = build_lm_batch(tokenizer, [format_prompt("What is the capital of France?")],
                       ["Paris."], max_length=64, pad_to="longest")
supervised = batch["labels"][0].ne(-100)
print(f"\nsequence tokens   {int(batch['attention_mask'][0].sum())}")
print(f"supervised tokens {int(supervised.sum())}  (response + EOS only)")
print("supervised text:  ", repr(tokenizer.decode(batch["input_ids"][0][supervised])))

## 7. Task 2: baseline sanity check

A few prompts through the untuned base model before committing to hours of
training - checkpoint path, tokenizer, prompt format, EOS behaviour, length.

Expect this to look bad. A pretrained base model continues text rather than
following instructions, and this one loops. On the probe run it answered
"2+2=4" and then repeated "Assistant" until it ran out of tokens. That is the
behaviour SFT is supposed to fix, and it is why the shared decoding policy
includes a repetition penalty.

In [ ]:
if RUN_GENERATION and Path(ALPACA_EVAL_PATH).exists():
    stage("baseline_sanity",
          ["python", "scripts/generate_responses.py",
           "--model_path", BASE_MODEL, "--prompts_path", ALPACA_EVAL_PATH,
           "--output_path", "outputs/eval/sanity/base_sanity.json",
           "--generator", "qwen2.5-0.5b-base", "--dataset", "alpaca_eval",
           "--limit", "3", "--batch_size", "3", "--resume", "false",
           "--max_new_tokens", str(MAX_NEW_TOKENS),
           "--repetition_penalty", str(REPETITION_PENALTY)],
          produces=["outputs/eval/sanity/base_sanity.json"],
          depends_on=["data_prep"], quiet_minutes=20)

    for row in read_json("outputs/eval/sanity/base_sanity.json"):
        print("=" * 78)
        print("PROMPT:", row["instruction"][:200])
        print("-" * 78)
        print("OUTPUT:", row["output"][:500] or "(empty)")

## 8. Task 3: supervised fine-tuning

Implements the nine requirements of section 9.1: prompt formatting,
response-only loss masking, padding and attention masks, gradient accumulation,
fp16 autocast, fp32 trainable weights under gradient scaling, gradient clipping,
validation loss, and best/final checkpoints.

| Hyperparameter | Value |
|---|---|
| Context length | 512 |
| Per-device batch x accumulation | 2 x 16 = 32 effective |
| Learning rate | 2e-5, cosine decay, 3% warmup |
| Weight decay / clipping | 0.1 / 1.0 |
| Epochs | 1 |
| Loss reduction | `token_mean` |

Checkpoints are written as fp16: six fp32 copies of a 0.5B model across the
three runs is ~12 GiB against a 19.5 GiB working disk. Training weights stay
fp32, which gradient scaling requires.

**Resume**: a checkpoint lands every 150 optimizer steps (~4 GiB with optimizer
state) and is deleted when the stage finishes. If this cell says `PAUSED`, rerun
the notebook with `PREV_OUTPUT` pointing at this run - it continues from the
last checkpoint.

In [ ]:
if RUN_SFT:
    stage("sft",
          ["python", "scripts/train_sft.py",
           "--model_name", BASE_MODEL,
           "--train_path", "data/processed/sft_train.json",
           "--val_path", "data/processed/sft_val.json",
           "--output_dir", "outputs/sft",
           "--max_length", "512",
           "--per_device_batch_size", "2", "--grad_accum_steps", "16",
           "--learning_rate", "2e-5", "--warmup_ratio", "0.03",
           "--weight_decay", "0.1", "--max_grad_norm", "1.0", "--epochs", "1",
           "--eval_every", str(SFT_EVAL_EVERY), "--eval_batch_size", "8",
           "--resume_every", str(SFT_RESUME_EVERY), "--seed", str(SEED),
           "--max_seconds", str(budget_left())],
          produces=["outputs/sft/best/model.safetensors", "outputs/sft/summary.json"],
          depends_on=["data_prep"])

    if Path("outputs/sft/summary.json").exists():
        print(json.dumps(read_json("outputs/sft/summary.json"), indent=2))
    disk_report()

## 9. Task 4: two from-scratch DPO runs

Both start from the same SFT best checkpoint, with a frozen copy of that same
checkpoint as the reference. Only the preference data differs.

| Hyperparameter | Value |
|---|---|
| beta | 0.1 |
| Learning rate | 1e-6, fixed (no decay) |
| Optimizer | RMSprop, as in the DPO paper |
| Per-device batch x accumulation | 1 x 32 = 32 effective |
| Context / max prompt | 512 / 256 |
| Checkpoint selection | highest validation reward accuracy |

Policy fp32 on `cuda:0`, frozen reference fp16 on `cuda:1`, as the manual's
memory tip suggests.

Watch the first evaluation line of each run. Policy and reference start
identical, so the loss is exactly log 2 = 0.6931, the margin is 0, and strict
reward accuracy is **0, not 0.5** - because `r_w > r_l` is false on a tie.

### 9a. HH-RLHF DPO (safety and refusal behaviour)

In [ ]:
if RUN_DPO_HH:
    stage("dpo_hh",
          ["python", "scripts/train_dpo.py", "--run_name", "dpo_hh",
           "--policy_path", "outputs/sft/best", "--reference_path", "outputs/sft/best",
           "--train_path", "data/processed/hh_train.json",
           "--val_path", "data/processed/hh_val.json",
           "--output_dir", "outputs/dpo_hh",
           "--beta", "0.1", "--learning_rate", "1e-6",
           "--optimizer", "rmsprop", "--lr_schedule", "constant",
           "--max_length", "512", "--max_prompt_length", "256",
           "--per_device_batch_size", "1", "--grad_accum_steps", "32",
           "--max_grad_norm", "1.0", "--epochs", "1",
           "--eval_every", str(DPO_EVAL_EVERY), "--eval_batch_size", "4",
           "--eval_subset", str(DPO_EVAL_SUBSET),
           "--policy_device", "cuda:0", "--reference_device", "cuda:1",
           "--resume_every", str(DPO_RESUME_EVERY), "--seed", str(SEED),
           "--max_seconds", str(budget_left())],
          produces=["outputs/dpo_hh/best/model.safetensors", "outputs/dpo_hh/summary.json"],
          depends_on=["sft"])

    if Path("outputs/dpo_hh/summary.json").exists():
        print(json.dumps(read_json("outputs/dpo_hh/summary.json"), indent=2))
    disk_report()

### 9b. UltraFeedback DPO (general helpfulness)

In [ ]:
if RUN_DPO_UF:
    stage("dpo_uf",
          ["python", "scripts/train_dpo.py", "--run_name", "dpo_uf",
           "--policy_path", "outputs/sft/best", "--reference_path", "outputs/sft/best",
           "--train_path", "data/processed/uf_train.json",
           "--val_path", "data/processed/uf_val.json",
           "--output_dir", "outputs/dpo_uf",
           "--beta", "0.1", "--learning_rate", "1e-6",
           "--optimizer", "rmsprop", "--lr_schedule", "constant",
           "--max_length", "512", "--max_prompt_length", "256",
           "--per_device_batch_size", "1", "--grad_accum_steps", "32",
           "--max_grad_norm", "1.0", "--epochs", "1",
           "--eval_every", str(DPO_EVAL_EVERY), "--eval_batch_size", "4",
           "--eval_subset", str(DPO_EVAL_SUBSET),
           "--policy_device", "cuda:0", "--reference_device", "cuda:1",
           "--resume_every", str(DPO_RESUME_EVERY), "--seed", str(SEED),
           "--max_seconds", str(budget_left())],
          produces=["outputs/dpo_uf/best/model.safetensors", "outputs/dpo_uf/summary.json"],
          depends_on=["sft"])

    if Path("outputs/dpo_uf/summary.json").exists():
        print(json.dumps(read_json("outputs/dpo_uf/summary.json"), indent=2))
    disk_report()

## 10. Task 5: TRL `DPOTrainer` + LoRA persona tuning

The one track where TRL and PEFT are allowed.

| Choice | Value | Why |
|---|---|---|
| Rank / alpha | 16 / 32 | ~200 pairs; a larger rank mostly adds variance. `alpha/r = 2` is standard. |
| Dropout | 0.05 | Light regularisation on a very small dataset. |
| Target modules | all attention + MLP projections | Style lives in token choice, not only attention. |
| Learning rate | 5e-5 | ~50x the full-finetune DPO rate: far fewer trainable weights. |
| Batch x accumulation | 2 x 4 = 8 | ~25 optimizer steps per epoch here. |
| Epochs | 3 | One epoch is too few steps to move style at this size. |
| Sequence length | 320 (prompt 128) | Measured: prompts reach ~33 tokens, responses ~60. Nothing is truncated. |

No explicit reference model: with a PEFT policy TRL scores the reference by
disabling the adapters, which halves memory.

TRL's constructor keywords move between releases. On the version here
(`trl 1.15`) `DPOTrainer` takes neither `beta` nor `max_length` - both live on
`DPOConfig` - and `DPOConfig` has no `max_prompt_length`. The builder inspects
the installed signatures and passes only what they accept, printing anything it
drops.

In [ ]:
if RUN_LORA_DPO:
    stage("lora_dpo",
          ["python", "scripts/train_lora_dpo.py",
           "--model_name", LORA_BASE_MODEL,
           "--train_path", "data/processed/persona_train.json",
           "--val_path", "data/processed/persona_val.json",
           "--output_dir", "outputs/lora_dpo_persona",
           "--beta", "0.1", "--learning_rate", "5e-5", "--epochs", str(LORA_EPOCHS),
           "--per_device_batch_size", "2", "--grad_accum_steps", "4",
           "--lora_r", "16", "--lora_alpha", "32", "--lora_dropout", "0.05",
           "--max_length", "320", "--max_prompt_length", "128",
           "--seed", str(SEED)],
          produces=["outputs/lora_dpo_persona/adapter",
                    "outputs/lora_dpo_persona/summary.json"],
          depends_on=["data_prep"])

    if Path("outputs/lora_dpo_persona/summary.json").exists():
        print(json.dumps(read_json("outputs/lora_dpo_persona/summary.json"), indent=2))

## 11. Training plots

SFT train/validation loss, and for each DPO run the loss plus reward accuracy and reward margin, with the selected checkpoint marked.

In [ ]:
stage("plots", ["python", "scripts/make_plots.py",
                "--outputs_dir", "outputs", "--plots_dir", "outputs/plots"],
      quiet_minutes=10)

from IPython.display import Image, display
for name in ["sft_loss.png", "dpo_hh_training.png", "dpo_uf_training.png",
             "lora_dpo_persona_training.png"]:
    path = Path("outputs/plots") / name
    if path.exists():
        print("=" * 78); print(name)
        display(Image(filename=str(path)))

## 12. Task 6: generation on the fixed evaluation sets

| Model | Evaluation set | Role |
|---|---|---|
| `qwen2.5-0.5b-base` | AlpacaEval 300 | Task 2 baseline, context only |
| `qwen2.5-0.5b-sft` | AlpacaEval 300 | Baseline for the UltraFeedback comparison |
| `qwen2.5-0.5b-dpo-uf` | AlpacaEval 300 | **Required comparison** |
| `qwen2.5-0.5b-sft` | HH eval 300 | Baseline for the HH comparison |
| `qwen2.5-0.5b-dpo-hh` | HH eval 300 | **Required comparison** |
| `smollm2-360m-instruct` | Persona style eval | Baseline for the persona track |
| `smollm2-360m-lora-dpo-persona` | Persona style eval | Persona result, reported separately |

Greedy, 256 new tokens, repetition penalty 1.1, applied identically to every
model in a comparison - otherwise the judge scores decoding policies rather
than checkpoints.

The persona pair uses `--chat_template tokenizer` rather than this project's
template. TRL formats conversational DPO data with the model's own template, and
SmolLM2-Instruct's template injects a default system message; generating without
it would score the LoRA model on a prompt shape it never saw in training.

Each file is rewritten after every batch and reruns resume, so a death here
costs one batch.

In [ ]:
GEN = "outputs/eval/generations"

def generate(tag, model_path, prompts_path, out_name, generator, dataset,
             adapter="", chat_template="project", depends=("sft",)):
    cmd = ["python", "scripts/generate_responses.py",
           "--model_path", model_path, "--prompts_path", prompts_path,
           "--output_path", f"{GEN}/{out_name}",
           "--generator", generator, "--dataset", dataset,
           "--batch_size", str(GEN_BATCH_SIZE),
           "--max_new_tokens", str(MAX_NEW_TOKENS),
           "--repetition_penalty", str(REPETITION_PENALTY),
           "--no_repeat_ngram_size", str(NO_REPEAT_NGRAM),
           "--chat_template", chat_template]
    if adapter:
        cmd += ["--adapter_path", adapter]
    return stage(f"gen_{tag}", cmd, produces=[f"{GEN}/{out_name}"],
                 depends_on=list(depends), quiet_minutes=25)

if RUN_GENERATION:
    # AlpacaEval: base (context), SFT (baseline), UltraFeedback DPO (candidate)
    generate("base_alpaca", BASE_MODEL, ALPACA_EVAL_PATH, "base_alpaca.json",
             "qwen2.5-0.5b-base", "alpaca_eval", depends=("data_prep",))
    generate("sft_alpaca", "outputs/sft/best", ALPACA_EVAL_PATH, "sft_alpaca.json",
             "qwen2.5-0.5b-sft", "alpaca_eval")
    generate("dpo_uf_alpaca", "outputs/dpo_uf/best", ALPACA_EVAL_PATH, "dpo_uf_alpaca.json",
             "qwen2.5-0.5b-dpo-uf", "alpaca_eval", depends=("dpo_uf",))

    # Helpful/harmless set: SFT baseline vs HH DPO
    generate("sft_hh", "outputs/sft/best", HH_EVAL_PATH, "sft_hh.json",
             "qwen2.5-0.5b-sft", "hh_rlhf_style_eval")
    generate("dpo_hh_hh", "outputs/dpo_hh/best", HH_EVAL_PATH, "dpo_hh_hh.json",
             "qwen2.5-0.5b-dpo-hh", "hh_rlhf_style_eval", depends=("dpo_hh",))

    # Persona track: the model's own chat template, for the reason above
    generate("persona_base", LORA_BASE_MODEL, PERSONA_EVAL_PATH, "persona_base.json",
             "smollm2-360m-instruct", "persona_style_eval",
             chat_template="tokenizer", depends=("data_prep",))
    generate("persona_lora", LORA_BASE_MODEL, PERSONA_EVAL_PATH, "persona_lora.json",
             "smollm2-360m-lora-dpo-persona", "persona_style_eval",
             adapter="outputs/lora_dpo_persona/adapter",
             chat_template="tokenizer", depends=("lora_dpo",))

### Generation statistics

DPO often changes verbosity, and a judge that likes long answers rewards that regardless of quality - so length belongs in the report next to the win rate.

In [ ]:
import statistics
rows = []
for path in sorted(Path(GEN).glob("*.json")):
    if path.name.endswith(".summary.json"):
        continue
    records = read_json(path)
    if not records:
        continue
    words = [len(r["output"].split()) for r in records]
    rows.append((records[0]["generator"], path.stem, len(records),
                 round(statistics.mean(words), 1), int(statistics.median(words)),
                 sum(1 for w in words if w == 0)))

header = f"{'generator':<32} {'set':<22} {'n':>4} {'mean':>6} {'median':>7} {'empty':>6}"
print(header); print("-" * len(header))
for r in rows:
    print(f"{r[0]:<32} {r[1]:<22} {r[2]:>4} {r[3]:>6} {r[4]:>7} {r[5]:>6}")

## 13. Blinded pairwise files

Responses are paired by prompt id and the side assignment is randomised per
prompt with seed 42. The position-to-model mapping lives in a `meta` block the
judge prompt never reads; verdicts are decoded back to identities only after
judging. A coin flip per prompt means position A is not a fixed model, so a
judge with a position bias cannot systematically favour one checkpoint.

In [ ]:
EVAL = "outputs/eval"

def pair(tag, baseline, candidate, out_name, depends):
    return stage(f"pairs_{tag}",
                 ["python", "scripts/build_pairwise.py",
                  "--sft_path", f"{GEN}/{baseline}", "--dpo_path", f"{GEN}/{candidate}",
                  "--output_path", f"{EVAL}/{out_name}", "--seed", str(SEED)],
                 produces=[f"{EVAL}/{out_name}"], depends_on=list(depends),
                 quiet_minutes=10)

pair("dpo_uf_vs_sft", "sft_alpaca.json", "dpo_uf_alpaca.json",
     "dpo_uf_vs_sft_pairs.jsonl", ["gen_sft_alpaca", "gen_dpo_uf_alpaca"])
pair("dpo_hh_vs_sft", "sft_hh.json", "dpo_hh_hh.json",
     "dpo_hh_vs_sft_pairs.jsonl", ["gen_sft_hh", "gen_dpo_hh_hh"])
pair("lora_persona", "persona_base.json", "persona_lora.json",
     "lora_persona_vs_base_pairs.jsonl", ["gen_persona_base", "gen_persona_lora"])

### Verify the blinding

Worth asserting rather than assuming: if a model name reaches the judge prompt, every number downstream is contaminated.

In [ ]:
from smalign.judge import build_batch_prompt
from smalign.utils import read_jsonl

FORBIDDEN = ["sft", "dpo", "qwen", "smollm", "lora", "a_model", "b_model",
             "checkpoint", "generator"]
found_any = False
for pairs_file in sorted(Path(EVAL).glob("*_pairs.jsonl")):
    found_any = True
    pairs = read_jsonl(pairs_file)
    prompt = build_batch_prompt(pairs[:5], safety_mode="hh" in pairs_file.name).lower()
    leaks = [t for t in FORBIDDEN if t in prompt]
    counts = {}
    for row in pairs:
        counts[row["meta"]["a_model"]] = counts.get(row["meta"]["a_model"], 0) + 1
    print(f"{pairs_file.name:<36} {len(pairs):>4} pairs  "
          f"{'LEAKS ' + ','.join(leaks) if leaks else 'blinded'}")
    print(f"{'':<36} position A: {counts}")
    assert not leaks, f"judge prompt for {pairs_file.name} leaks {leaks}"
print("\nall judge prompts are blinded" if found_any else "no pairwise files yet")

### Read the generations before trusting any number

In [ ]:
def compare(left_path, right_path, n=2, width=500, only_category=None):
    if not (Path(left_path).exists() and Path(right_path).exists()):
        print(f"(missing {left_path} or {right_path})"); return
    left = {r["id"]: r for r in read_json(left_path)}
    right = {r["id"]: r for r in read_json(right_path)}
    shown = 0
    for row_id, a in left.items():
        if row_id not in right: continue
        if only_category and a.get("category") != only_category: continue
        b = right[row_id]
        print("=" * 78)
        print(f"{row_id}{'  [' + a['category'] + ']' if a.get('category') else ''}")
        print(a["instruction"][:250]); print("-" * 78)
        print(f"[{a['generator']}]\n{a['output'][:width] or '(empty)'}"); print("-" * 78)
        print(f"[{b['generator']}]\n{b['output'][:width] or '(empty)'}"); print()
        shown += 1
        if shown >= n: break

print("#" * 78); print("# UltraFeedback DPO vs SFT on AlpacaEval"); print("#" * 78)
compare(f"{GEN}/sft_alpaca.json", f"{GEN}/dpo_uf_alpaca.json")
print("#" * 78); print("# HH DPO vs SFT: a harmful request (refusal)"); print("#" * 78)
compare(f"{GEN}/sft_hh.json", f"{GEN}/dpo_hh_hh.json", only_category="harmful_instructions")
print("#" * 78); print("# HH DPO vs SFT: a benign request (over-refusal check)"); print("#" * 78)
compare(f"{GEN}/sft_hh.json", f"{GEN}/dpo_hh_hh.json", only_category="benign_helpfulness")
print("#" * 78); print("# Persona LoRA vs base: safety-sensitive"); print("#" * 78)
compare(f"{GEN}/persona_base.json", f"{GEN}/persona_lora.json", only_category="safety_sensitive")

## 14. Status and handover

The table below is the first thing to read when this run comes back. Anything
`PAUSED` means the stage stopped on the time budget with a checkpoint written -
save this version, attach it as `PREV_OUTPUT`, and run again.

In [ ]:
print_ledger(ledger)
disk_report()

print("\npairwise files for the judge notebook:")
for path in sorted(Path(EVAL).glob("*_pairs.jsonl")):
    print(f"  {len(read_jsonl(path)):>4} pairs   {path}")

# Small artifacts only. Checkpoints are preserved by the saved version itself;
# zipping them too would double their footprint on a disk that is already tight.
!rm -f /kaggle/working/pipeline_artifacts.zip
!cd /kaggle/working/small-model-alignment && zip -q -r /kaggle/working/pipeline_artifacts.zip \
    outputs/plots outputs/eval outputs/*/metrics.jsonl outputs/*/summary.json \
    outputs/*/run_config.json outputs/pipeline_state.json configs \
    data/processed/data_manifest.json eval_sets 2>/dev/null
!ls -la /kaggle/working/pipeline_artifacts.zip 2>/dev/null

done = all(ledger.is_done(s) for s in
           ["data_prep", "sft", "dpo_hh", "dpo_uf", "lora_dpo",
            "pairs_dpo_uf_vs_sft", "pairs_dpo_hh_vs_sft"])
print("\n" + ("ALL STAGES COMPLETE -> go to 02_judge_kaggle.ipynb"
              if done else "SOME STAGES ARE INCOMPLETE -> rerun with PREV_OUTPUT set to this version"))

---

## Next steps

1. This notebook's version is saved automatically by a committed run. If any
   stage shows `PAUSED` or `FAILED`, start a new run of this same notebook with
   `PREV_OUTPUT` set to this version's output - finished stages are skipped and
   paused ones continue from their checkpoints.
2. When every stage is complete, open **`02_judge_kaggle.ipynb`**, created from
   <https://www.kaggle.com/benchmarks/tasks/new>, and attach this output. It
   reads the pairwise files and runs the model-based judging.
3. Fill in `report/REPORT_TEMPLATE.md` from `outputs/results_summary.md`,
   `outputs/plots/`, and the `run_config.json` of each run.